# Telenet Mailbox Scanner — Distinct Sender/Receiver List

This notebook connects to your **stijn.huysman@telenet.be** mailbox over IMAP and:

1. **Discovers every folder, including nested subfolders**, automatically
2. Scans message **headers only** (From/To/Cc/Subject/Date/List-Unsubscribe — never the message body) across all of them
3. Builds **one distinct table** of every email address you've ever sent to or received from — each address is added **once**, the first time it's seen; later messages from/to the same address are skipped rather than updating the row
4. Flags rows that look like a **subscription/service** (via `List-Unsubscribe` or sender patterns like `noreply@`)
5. Exports a single Excel file: `contacts.xlsx`

## Before you run it
- Run this on your own PC — your password is typed locally via a hidden prompt and only goes to Telenet's IMAP server over SSL. Nothing is sent anywhere else.
- If Telenet ever asks you to generate a dedicated **app password** for mail clients, use that instead of your normal webmail password.
- `MAX_MESSAGES_PER_FOLDER` caps each folder while you're testing; raise it (or set to `None`) once you've validated the output.


In [10]:
import imaplib
import email
from email.header import decode_header
from email.utils import getaddresses, parsedate_to_datetime
import getpass
import re
import time
import pandas as pd

IMAP_SERVER = "imap.telenet.be"
IMAP_PORT = 993
EMAIL_ADDRESS = "stijn.huysman@telenet.be"

# Safety cap per folder while testing. None = scan every message in every folder (no cap).
# Leaving this set to a number (e.g. 500) silently skips everyone who only ever emailed you
# in older messages beyond that cutoff - set to None once you're ready for the real run.
MAX_MESSAGES_PER_FOLDER = None

# Folders whose name contains any of these (case-insensitive) are treated as "outgoing" mail,
# i.e. where the To/Cc addresses are people YOU contacted rather than people who contacted you.
SENT_FOLDER_KEYWORDS = ["sent", "verzonden"]

# No folders excluded - every folder and nested subfolder (incl. Trash/Junk/Spam/Deleted) is scanned.
EXCLUDE_FOLDER_KEYWORDS = []


In [11]:
EMAIL_PASSWORD = getpass.getpass("Telenet email password (hidden input): ")

imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
imap.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
print("Connected as", EMAIL_ADDRESS)


Connected as stijn.huysman@telenet.be


## Step 1 — Discover all folders (including nested subfolders)

IMAP's `LIST` command returns *every* folder in the account, nested ones included — a subfolder just shows up with its full path (e.g. `Inbox/Newsletters` or `Inbox.Newsletters`, depending on the server's hierarchy separator). The parsing below extracts the real folder name regardless of that separator.

In [12]:
def parse_folder_name(raw_line):
    """Parse a single line returned by imap.list() into a real folder name."""
    decoded = raw_line.decode(errors="replace") if isinstance(raw_line, bytes) else raw_line
    match = re.search(r'"([^"]*)"\s*$', decoded)
    if match:
        return match.group(1)
    return decoded.split(" ")[-1].strip('"')


def discover_folders(imap_conn):
    status, folders_raw = imap_conn.list()
    if status != "OK":
        raise RuntimeError("Could not list folders")
    all_folders = [parse_folder_name(f) for f in folders_raw]
    excluded = [
        f for f in all_folders
        if any(kw.lower() in f.lower() for kw in EXCLUDE_FOLDER_KEYWORDS)
    ]
    included = [f for f in all_folders if f not in excluded]
    return included, excluded


included_folders, excluded_folders = discover_folders(imap)
print(f"Found {len(included_folders)} folders to scan (incl. nested subfolders):")
for f in included_folders:
    tag = "  [SENT-type]" if any(kw in f.lower() for kw in SENT_FOLDER_KEYWORDS) else ""
    print(f"  - {f}{tag}")

if excluded_folders:
    print(f"\nSkipping {len(excluded_folders)} folder(s) based on EXCLUDE_FOLDER_KEYWORDS:")
    for f in excluded_folders:
        print(f"  - {f}")


Found 48 folders to scan (incl. nested subfolders):
  - 0 geclasseerd
  - 0 geclasseerd/MASTAT
  - 0 geclasseerd/smartschool
  - Archief
  - Archive
  - Drafts
  - Gutami
  - INBOX
  - INBOX/0 VERKOOP
  - INBOX/0. Antoon
  - INBOX/0. Antoon/medeleven
  - INBOX/0. Antoon/tiptop
  - INBOX/0. Antoon/total
  - INBOX/0. Appostolinen
  - INBOX/0. arc
  - INBOX/0. Basket
  - INBOX/0. Basket/refs
  - INBOX/0. CECILE
  - INBOX/00. Koppenberg
  - INBOX/00. LIMA
  - INBOX/00. mercedes
  - INBOX/00.reis
  - INBOX/1. Coupure verhuur
  - INBOX/1. Coupure verhuur/1a Coupure
  - INBOX/5 persoonlijk
  - INBOX/5 persoonlijk/Marijke
  - INBOX/5 persoonlijk/zonnepanelen
  - INBOX/7 opleiding
  - INBOX/andere
  - INBOX/andere/bouw H135
  - INBOX/andere/Christeyns
  - INBOX/andere/familie
  - INBOX/andere/fiscaal attest
  - INBOX/andere/rekeningne
  - INBOX/andere/vrienden
  - INBOX/kot Anna
  - INBOX/mercedes
  - INBOX/verzekering
  - INBOX/zonnepanelen
  - Junk
  - marijke
  - Prullenbak
  - Sent  [SENT-t

If a folder name looks wrong above, override the list manually before continuing:
```python
included_folders = ["INBOX", "INBOX/Newsletters", "Verzonden Items", ...]
```

## Step 2 — Distinct-list tracker

This is the core of the "only add it if it's not already in the list" behaviour: `known_addresses` is a dict keyed by email address. Every header we look at is checked against it — if the address is already known, we skip it and move on immediately without touching that row again. Only genuinely new addresses get added, and each one is written to the table exactly once, using the details from the message where it was first spotted.

In [13]:
AUTOMATED_PATTERNS = re.compile(
    r"(no.?reply|newsletter|notification|donotreply|mailer|updates?@|info@|support@|alerts?@|marketing@)",
    re.IGNORECASE,
)

def decode_str(value):
    """Decode a possibly-encoded email header into plain text."""
    if not value:
        return ""
    parts = decode_header(value)
    out = []
    for text, enc in parts:
        if isinstance(text, bytes):
            try:
                out.append(text.decode(enc or "utf-8", errors="replace"))
            except LookupError:
                out.append(text.decode("utf-8", errors="replace"))
        else:
            out.append(text)
    return "".join(out)


def looks_like_subscription(sender_email, list_unsubscribe):
    if list_unsubscribe:
        return True
    if AUTOMATED_PATTERNS.search(sender_email or ""):
        return True
    return False


# email -> row dict. Populated only the first time each address is encountered.
known_addresses = {}


def add_if_new(addr_email, addr_name, role, folder, date, likely_subscription):
    """Add addr_email to known_addresses only if it isn't already there."""
    addr_email = (addr_email or "").strip().lower()
    if not addr_email or addr_email == EMAIL_ADDRESS.lower():
        return
    if addr_email in known_addresses:
        return  # already in the list - skip, do not update
    known_addresses[addr_email] = {
        "email": addr_email,
        "name": addr_name or "",
        "role_first_seen_as": role,          # "Sender" or "Receiver"
        "likely_subscription": likely_subscription,
        "first_seen_folder": folder,
        "first_seen_date": date,
    }


## Step 3 — Scan every folder, header-only, feeding the tracker

In [14]:
skipped_folders = []  # (folder_name, reason) - so a silent failure never looks like "zero new addresses"


def reconnect():
    """Re-establish the IMAP connection after the server drops it (e.g. Zimbra abort)."""
    global imap
    try:
        imap.logout()
    except Exception:
        pass
    imap = imaplib.IMAP4_SSL(IMAP_SERVER, IMAP_PORT)
    imap.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    print("  Reconnected.")


def process_folder(imap_conn, folder_name, max_messages=None, retries_left=3):
    try:
        status, _ = imap_conn.select(f'"{folder_name}"', readonly=True)
        if status != "OK":
            skipped_folders.append((folder_name, "could not open (select failed)"))
            print(f"  Could not open folder: {folder_name} (skipping)")
            return 0

        status, data = imap_conn.search(None, "ALL")
        if status != "OK":
            skipped_folders.append((folder_name, "search failed"))
            print(f"  Search failed on folder: {folder_name} (skipping)")
            return 0
    except (imaplib.IMAP4.abort, imaplib.IMAP4.error, OSError) as e:
        if retries_left <= 0:
            skipped_folders.append((folder_name, f"connection kept dropping: {e}"))
            print(f"  Giving up on {folder_name} after repeated connection drops: {e}")
            return 0
        print(f"  Connection dropped opening {folder_name} ({e}) - reconnecting and retrying...")
        reconnect()
        time.sleep(2)
        return process_folder(imap, folder_name, max_messages, retries_left - 1)

    ids = data[0].split()
    if not ids:
        return 0
    if max_messages:
        ids = ids[-max_messages:]

    print(f"  {folder_name}: {len(ids)} messages to scan")

    is_sent_type = any(kw in folder_name.lower() for kw in SENT_FOLDER_KEYWORDS)
    header_fields = "(FROM TO CC SUBJECT DATE LIST-UNSUBSCRIBE)"
    # Smaller batches than before - large batch fetches are more likely to trip Zimbra's
    # connection limits and cause it to abort the connection.
    batch_size = 50
    new_count = 0

    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        id_set = b",".join(batch)
        try:
            status, msg_data = imap.fetch(id_set, f"(BODY.PEEK[HEADER.FIELDS {header_fields}])")
        except (imaplib.IMAP4.abort, imaplib.IMAP4.error, OSError) as e:
            if retries_left <= 0:
                skipped_folders.append((folder_name, f"fetch kept dropping at batch {i}: {e}"))
                print(f"  Giving up on a batch in {folder_name} after repeated drops: {e}")
                continue
            print(f"  Connection dropped mid-fetch in {folder_name} ({e}) - reconnecting and retrying this folder...")
            reconnect()
            time.sleep(2)
            # Re-select the folder after reconnecting (a fresh connection has no folder selected)
            # and retry the WHOLE folder from scratch - simplest safe option, addresses already
            # in known_addresses are skipped again instantly so this doesn't double-count anything.
            return process_folder(imap, folder_name, max_messages, retries_left - 1)

        if status != "OK":
            skipped_folders.append((folder_name, f"fetch failed for batch starting at {i}"))
            continue
        for part in msg_data:
            if not isinstance(part, tuple):
                continue
            msg = email.message_from_bytes(part[1])
            from_addr = getaddresses([msg.get("From", "")])
            to_addrs = getaddresses([msg.get("To", "")])
            cc_addrs = getaddresses([msg.get("Cc", "")])
            date_raw = msg.get("Date", "")
            try:
                date_parsed = parsedate_to_datetime(date_raw) if date_raw else None
            except Exception:
                date_parsed = None
            list_unsub = msg.get("List-Unsubscribe", "")

            before = len(known_addresses)

            if is_sent_type:
                # In a sent-type folder, the addresses you contacted are the recipients
                for name, addr in to_addrs + cc_addrs:
                    add_if_new(addr, decode_str(name), "Receiver", folder_name, date_parsed, False)
            else:
                # Elsewhere, the sender is who contacted you
                if from_addr:
                    name, addr = from_addr[0]
                    sub_flag = looks_like_subscription(addr.lower() if addr else "", list_unsub)
                    add_if_new(addr, decode_str(name), "Sender", folder_name, date_parsed, sub_flag)

            new_count += len(known_addresses) - before

    return new_count


print("Scanning folders...")
total_new = 0
for folder in included_folders:
    total_new += process_folder(imap, folder, MAX_MESSAGES_PER_FOLDER)
    time.sleep(0.5)  # small pause between folders - eases sustained load on the server

print(f"\nDistinct addresses collected so far: {len(known_addresses)}")

if skipped_folders:
    print(f"\nWARNING: {len(skipped_folders)} folder/batch operation(s) failed and were skipped - "
          f"any addresses only present there are MISSING from the list below:")
    for name, reason in skipped_folders:
        print(f"  - {name}: {reason}")
else:
    print("\nNo folders or batches failed - every included folder was fully scanned.")


Scanning folders...
  0 geclasseerd: 2330 messages to scan
  0 geclasseerd/MASTAT: 3 messages to scan
  Archief: 404 messages to scan
  Archive: 9 messages to scan
  Gutami: 61 messages to scan
  INBOX: 143 messages to scan
  INBOX/0 VERKOOP: 131 messages to scan
  INBOX/0. Antoon: 558 messages to scan
  INBOX/0. Antoon/medeleven: 5 messages to scan
  INBOX/0. Antoon/tiptop: 9 messages to scan
  INBOX/0. Antoon/total: 6 messages to scan
  INBOX/0. Appostolinen: 14 messages to scan
  INBOX/0. arc: 528 messages to scan
  INBOX/0. Basket: 168 messages to scan
  INBOX/0. Basket/refs: 9 messages to scan
  INBOX/0. CECILE: 17 messages to scan
  INBOX/00. Koppenberg: 9 messages to scan
  INBOX/00. LIMA: 10 messages to scan
  INBOX/00. mercedes: 4 messages to scan
  INBOX/00.reis: 1 messages to scan
  INBOX/1. Coupure verhuur: 89 messages to scan
  INBOX/1. Coupure verhuur/1a Coupure: 41 messages to scan
  INBOX/5 persoonlijk: 3402 messages to scan
  Connection dropped mid-fetch in INBOX/5 per

In [15]:
imap.logout()
print("Done fetching, connection closed.")


Done fetching, connection closed.


## Step 4 — Build and inspect the distinct table

In [16]:
contacts_df = pd.DataFrame(known_addresses.values()).sort_values(
    ["likely_subscription", "role_first_seen_as", "email"]
)
print(f"Total distinct addresses: {len(contacts_df)}")
contacts_df


Total distinct addresses: 2684


,email,name,role_first_seen_as,likely_subscription,first_seen_folder,first_seen_date
2345,21b1633c-7b14-4b87-b021-d942df33bf38@reply.lin...,Emily Desmadrille,Receiver,False,Sent,2022-09-19 16:53:45+02:00
2081,35959e25-9ddf-4f44-b844-0f138d1687ce@reply.lin...,Bruno Baetens,Receiver,False,Sent,2017-06-05 23:00:49+02:00
1950,45438930_fdedbfdb-ab34-404f-a444-d12d4d8e31ae@...,Sofie Verpoucke (via LinkedIn),Receiver,False,Sent,2013-03-27 12:22:36+01:00
1965,4rno@live.be,,Receiver,False,Sent,2014-02-20 21:42:09+01:00
2352,71cce0c3-1c16-489c-ba3c-bbf900142c10@reply.lin...,Gareth Newall,Receiver,False,Sent,2022-10-27 09:07:14+02:00
...,...,...,...,...,...,...
1559,webinar@gezinsbond.be,Gezinsbond,Sender,True,INBOX/5 persoonlijk,2017-09-20 14:19:43+02:00
402,webinar@neo4j.com,"Jonathan Thein, Neo4j",Sender,True,0 geclasseerd,2023-06-07 03:02:03-05:00
1570,webmaster@mm.cm.be,CM,Sender,True,INBOX/5 persoonlijk,2017-10-20 19:19:32+02:00
373,website@myonlinetraininghub.com,My Online Training Hub,Sender,True,0 geclasseerd,2023-04-13 13:07:57+00:00


## Step 5 — Export to Csv

In [17]:
contacts_df.to_csv("contacts.csv", index=False)
print("Saved contacts.csv in the current folder.")


Saved contacts.csv in the current folder.


## Notes / tuning tips

- **Distinct by design**: `known_addresses` is a plain dict keyed by lowercased email address. `add_if_new()` checks membership before doing anything else, so an address that already made it into the table is never touched again — no matter how many more messages from/to it are found afterwards.
- **`role_first_seen_as`** reflects only the *first* message where the address appeared, not every appearance — e.g. if you emailed someone once and they'd already emailed you ten times before, they'll show as "Sender" (the earlier, first-seen role), since that folder was processed first in `included_folders` order. If you want strictly chronological first-seen regardless of folder order, sort `included_folders` so folders are scanned oldest-activity-first, or add a global date comparison inside `add_if_new`.
- **`likely_subscription`**: True if the message had a `List-Unsubscribe` header or the sender matches common automated patterns (`noreply@`, `newsletter@`, etc.). Only set for entries first seen as a Sender — worth double-checking manually since some services don't set these signals.
- **Nested folders** are included automatically via the `LIST` command's full folder paths.
- **Rate limiting**: if Telenet's IMAP server throttles or disconnects on very large mailboxes, reduce `batch_size` inside `process_folder` (e.g. to 50) and rerun.
